In [ ]:
import pandas as pd

# Goals
- Benchmark the use of controlled vocabularies in the Keyword and Topic Classification fields in the first versions of the first dataset of first-time users who publish in the "Root" collection or in a collection managed by those users
    - Import and prepare information about datasets queried from the copy of the Harvard Dataverse database
    - Get counts of datasets that have values in one or more Keywords or Topic Classification fields
    - Get counts of those datasets where the values in the Keywords or Topic Classification fields come from a controlled vocabulary
        - Group counts of these datasets into 90 day periods
    - Get the list of the controlled vocabularies that these values come from and how many datasets use these controlled vocabularies
- Learn how often users have entered term URIs in the Keyword Controlled Vocabulary URL field or in the Topic Classification Controlled Vocabulary URL field

## Import and prepare information queried from the copy of the Harvard Dataverse database about the first versions of the first dataset of first-time users who publish in the "Root" collection or in a collection managed by those users

In [ ]:
# Import metadata about all datasets
datasetInfoDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2024.08.25_03.34.11/dataset_pids_from_most_known_dataverse_installations_2024.08.csv',
    usecols=lambda x: x in [
        'dataset_pid_url', 'dataverse_installation_name',
        'dataverse_collection_name', 'dataverse_collection_alias',
        'dataverse_collection_type'],
    low_memory=False
)

# Import keyword metadata
keywordsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2024.08.25_03.34.11/csv_files_with_metadata_from_most_known_dataverse_installations/keyword_2024.08.25-2024.08.30.csv',
    usecols=lambda x: x in [
        'dataset_pid_url', 'dataset_publication_date',
        'keywordValue', 'keywordVocabulary',
        'keywordVocabularyURI', 'keywordTermURI']
)

# Import topic classification metadata
topicClassificationDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2024.08.25_03.34.11/csv_files_with_metadata_from_most_known_dataverse_installations/topic_classification_2024.08.25-2024.08.30.csv',
    usecols=lambda x: x in [
        'dataset_pid_url', 'dataset_publication_date',
        'topicClassValue', 'topicClassVocab',
        'topicClassVocabURI'])

# Merge the topicClassificationDf and keywordsDf dataframes
keywordsAndTopicClassDf = keywordsDf.merge(
    topicClassificationDf, how='outer',
    on=['dataset_pid_url', 'dataset_publication_date'])

In [ ]:
# Include only datasets where a user wanted to use a MeSH term in a keyword or topic classification field
meshBaseUrls = ['nlm.nih.gov/mesh', 'meshb.nlm.nih.gov']
meshKeywordsAndTopicClassDf = (keywordsAndTopicClassDf
    .query(
        'keywordValue.str.contains("|".join(@meshBaseUrls)) or\
        keywordVocabulary.str.contains("mesh")==True or\
        keywordVocabularyURI.str.contains("|".join(@meshBaseUrls)) or\
        keywordTermURI.str.contains("|".join(@meshBaseUrls)) or\
        topicClassVocab.str.contains("mesh")==True or\
        topicClassValue.str.contains("|".join(@meshBaseUrls)) or\
        topicClassVocabURI.str.contains("|".join(@meshBaseUrls))',
        engine="python"
    )
)

# Merge meshKeywordsAndTopicClassDf with datasetInfoDf so we see where these datasets are published
meshKeywordsAndTopicClassDf = meshKeywordsAndTopicClassDf.merge(
    datasetInfoDf,
    how='inner',
    on=['dataset_pid_url'])

# Reorder columns
meshKeywordsAndTopicClassDf = meshKeywordsAndTopicClassDf[[
    'dataverse_installation_name',
    'dataverse_collection_name',
    'dataverse_collection_alias',
    'dataverse_collection_type',
    'dataset_pid_url',
    'dataset_publication_date',
    'keywordValue',
    'keywordVocabulary',
    'keywordTermURI',
    'keywordVocabularyURI',
    'topicClassValue',
    'topicClassVocab',
    'topicClassVocabURI'
]]

# Check dataframe
meshKeywordsAndTopicClassDf.info()

In [ ]:
# Export dataframe to CSV file
meshKeywordsAndTopicClassDf.to_csv('/Users/juliangautier/Desktop/meshKeywordsAndTopicClassDf.csv', index=False)

# Explore meshKeywordsAndTopicClassDf

- How many datasets are there and which installations publish them?
- In Harvard Dataverse, which collections contain datasets where MeSH terms are used in the keyword and topic classification fields?
- In Harvard Dataverse, which datasets outside of collections include MeSH terms?

In [ ]:
# How many datasets are there and which installations publish them?
datasetCount = len(pd.unique(meshKeywordsAndTopicClassDf['dataset_pid_url']))
print(datasetCount)

In [ ]:
# Count of datasets by installation
installationCount = len(pd.unique(meshKeywordsAndTopicClassDf['dataverse_installation_name']))
print(installationCount)
datasetCountByInstallation = (
    meshKeywordsAndTopicClassDf
        [['dataverse_installation_name', 'dataset_pid_url']]
        .drop_duplicates()
        .value_counts(subset=['dataverse_installation_name'])
        .to_frame('count')
        .reset_index(drop=False, inplace=False)
    )
datasetCountByInstallation.head(installationCount)

In [ ]:
# In Harvard Dataverse, which collections contain datasets where MeSH terms are used?
hdvMeshKeywordsAndTopicClassDf = (meshKeywordsAndTopicClassDf
    .query(
        'dataverse_installation_name == "Harvard Dataverse"'
    )
)
hdvMeshKeywordsAndTopicClassDf.info()
# hdvMeshKeywordsAndTopicClassDf.to_csv('/Users/juliangautier/Desktop/hdvMeshKeywordsAndTopicClassDf.csv', index=False)

In [ ]:
collectionCount = len(pd.unique(meshKeywordsAndTopicClassDf['dataverse_collection_alias']))
print(installationCount)

datasetCountByCollection = (
    hdvMeshKeywordsAndTopicClassDf
    [['dataverse_collection_alias', 'dataset_pid_url']]
        .drop_duplicates()
        .value_counts(subset=['dataverse_collection_alias'])
        .to_frame('count')
        .reset_index(drop=False, inplace=False)
)
datasetCountByCollection.head(collectionCount)

## Use of keyword fields versus topic classification fields

In [ ]:
keywordsAndTopicClassinSameDatasetDf = (keywordsAndTopicClassDf
    .query(
        '(keywordValue == keywordValue or keywordValue.notnull()) and\
        (topicClassValue == topicClassValue or topicClassValue.notnull())',
    engine="python"
    )
)
print(len(keywordsAndTopicClassinSameDatasetDf))

In [ ]:
# Merge keywordsAndTopicClassinSameDatasetDf with datasetInfoDf so we see where these datasets are published
keywordsAndTopicClassinSameDatasetDf = keywordsAndTopicClassinSameDatasetDf.merge(
    datasetInfoDf,
    how='inner',
    on=['dataset_pid_url'])
print(len(keywordsAndTopicClassinSameDatasetDf))

In [ ]:
keywordsAndTopicClassinSameDatasetDf.info()

In [ ]:
keywordsAndTopicClassinSameDatasetDf.to_csv('/Users/juliangautier/Desktop/keywordsAndTopicClassinSameDatasetDf.csv', index=False)

In [ ]:
installationCount = len(pd.unique(keywordsAndTopicClassinSameDatasetDf['dataverse_installation_name']))
print(installationCount)